# 🍎 Phi-4 Model with AIProjectClient 🍏

**Phi-4** is a next-generation open model that aims to provide near GPT-4o capabilities at a fraction of the cost, making it ideal for many enterprise or personal use cases. It's especially great for chain-of-thought reasoning and RAG (Retrieval Augmented Generation) scenarios.

In this notebook, you'll see how to:
1. **Initialize** an `AIProjectClient` for your Azure AI Foundry environment.
2. **Chat** with the **Phi-4** model using `azure-ai-inference`.
3. **Show** a Health & Fitness example, featuring disclaimers and wellness Q&A.
4. **Enjoy** the value proposition of a cheaper alternative to GPT-4 with strong reasoning capabilities. 🏋️

> **Disclaimer**: This is not medical advice. Please consult professionals.

## Why Phi-4?
Phi-4 is a 14B-parameter model trained on curated data for high reasoning performance.
- **Cost-Effective**: Get GPT-4-level performance for many tasks without the GPT-4 price.
- **Reasoning & RAG**: Perfect for chain-of-thought reasoning steps and retrieval augmented generation workflows.
- **Generous Context Window**: 16K tokens, enabling more context or longer user conversations.

<img src="./seq-diagrams/4-phi-4.png" width="30%"/>


## 1. Setup

Below, we'll install and import the necessary libraries:
- **azure-ai-projects**: For the `AIProjectClient`.
- **azure-ai-inference**: For calling your model, specifically the chat completions.
- **azure-identity**: For `DefaultAzureCredential`.

Ensure you have a `.env` file with:
```bash
PROJECT_ENDPOINT=<your-project-endpoint>
MICROSOFT_MODEL=phi-4
```

> **Note**: It's recommended to complete the [`3-basic-rag.ipynb`](./3-basic-rag.ipynb) notebook before this one, as it covers important concepts that will be helpful here.

In [ ]:
import os
import requests
from dotenv import load_dotenv
from pathlib import Path
from urllib.parse import urlparse
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Load environment variables from workspace root .env
notebook_path = Path().absolute()
load_dotenv(notebook_path.parent.parent / '.env')

# Initialize credentials
credential = AzureCliCredential()

# Parse PROJECT_ENDPOINT into required AIProjectClient constructor components
_url            = os.getenv("PROJECT_ENDPOINT")
_parsed         = urlparse(_url)
base_endpoint   = f"{_parsed.scheme}://{_parsed.netloc}"
path_parts      = [p for p in _parsed.path.split("/") if p]
project_name    = path_parts[-1] if path_parts else ""
hub_name        = _parsed.netloc.split(".")[0]
phi4_model      = os.getenv("MICROSOFT_MODEL", "Phi-4")

# Auto-detect subscription_id & resource_group from Foundry hub
print("Auto-detecting subscription ID and resource group...")
try:
    mgmt_token = credential.get_token("https://management.azure.com/.default").token
    headers = {"Authorization": f"Bearer {mgmt_token}"}

    subs = requests.get(
        "https://management.azure.com/subscriptions?api-version=2020-01-01",
        headers=headers, timeout=15
    ).json().get("value", [])

    subscription_id = None
    resource_group = None

    for sub in subs:
        sub_id = sub["subscriptionId"]
        resources = requests.get(
            f"https://management.azure.com/subscriptions/{sub_id}/resources"
            f"?$filter=name eq '{hub_name}' and "
            f"resourceType eq 'Microsoft.CognitiveServices/accounts'"
            f"&api-version=2021-04-01",
            headers=headers, timeout=15
        ).json().get("value", [])

        if resources:
            resource_group = resources[0]["id"].split("/")[4]
            subscription_id = sub_id
            break

    if not (subscription_id and resource_group):
        raise RuntimeError(f"Hub '{hub_name}' not found in any subscription.")

    print(f"✓ Subscription ID:  {subscription_id[:8]}...")
    print(f"✓ Resource group:   {resource_group}")

except Exception as e:
    raise EnvironmentError(f"Failed to auto-detect subscription and resource group: {e}") from e

try:
    project_client = AIProjectClient(
        endpoint=base_endpoint,
        subscription_id=subscription_id,
        resource_group_name=resource_group,
        project_name=project_name,
        credential=credential,
    )
    print("✅ AIProjectClient created successfully!")
except Exception as e:
    print("❌ Error creating AIProjectClient:", e)

# Phi-4 is an Azure AI catalog model — use the /models inference endpoint, NOT /openai/v1
phi4_endpoint = f"{base_endpoint}/models"
phi4_client = ChatCompletionsClient(
    endpoint=phi4_endpoint,
    credential=AzureKeyCredential(os.getenv("AZURE_OPENAI_KEY"))
)
print(f"✅ Phi-4 client initialized | Endpoint: {phi4_endpoint} | Model: {phi4_model}")

## 2. Chat with Phi-4 🍏
We'll demonstrate a simple conversation using **Phi-4** in a health & fitness context. We'll define a system prompt that clarifies the role of the assistant. Then we'll ask some user queries.

> Notice that Phi-4 is well-suited for chain-of-thought reasoning. We'll let it illustrate its reasoning steps for fun.


In [ ]:
def chat_with_phi4(user_question, chain_of_thought=False):
    """Send a chat request to the Phi-4 model with optional chain-of-thought."""
    system_prompt = (
        "You are a Phi-4 AI assistant, focusing on health and fitness.\n"
        "Remind users that you are not a medical professional, but can provide general info.\n"
    )

    if chain_of_thought:
        system_prompt += "Please show your step-by-step reasoning in your answer.\n"

    response = phi4_client.complete(
        model=phi4_model,
        messages=[
            SystemMessage(content=system_prompt),
            UserMessage(content=user_question),
        ],
        temperature=0.8,
        top_p=0.9,
        max_tokens=400,
    )

    return response.choices[0].message.content

# Example usage:
question = "I'm training for a 5K. Any tips on a weekly workout schedule?"
answer = chat_with_phi4(question, chain_of_thought=True)
print("🗣️ User:", question)
print("🤖 Phi-4:", answer)

## 3. RAG-like Example (Stub)
Phi-4 also excels in retrieval augmented generation scenarios, where you provide external context and let the model reason over it. Below is a **stub** example showing how you'd pass retrieved text as context.

> In a real scenario, you'd embed & search for relevant passages, then feed them into the user/system message.


In [ ]:
def chat_with_phi4_rag(user_question, retrieved_doc):
    """Simulate an RAG flow by appending retrieved context to the system prompt."""
    system_prompt = (
        "You are Phi-4, helpful fitness AI.\n"
        "We have some context from the user's knowledge base:\n"
        f"{retrieved_doc}\n"
        "Please use this context to help your answer. If the context doesn't help, say so.\n"
    )

    response = phi4_client.complete(
        model=phi4_model,
        messages=[
            SystemMessage(content=system_prompt),
            UserMessage(content=user_question),
        ],
        temperature=0.3,
        max_tokens=300,
    )
    return response.choices[0].message.content

# Define a doc snippet for RAG demo
doc_snippet = (
    "Recommended to run 3 times per week and mix with cross-training.\n"
    "Include rest days or active recovery days for muscle repair."
)

user_q = "How often should I run weekly to prepare for a 5K?"
rag_answer = chat_with_phi4_rag(user_q, doc_snippet)
print("🗣️ User:", user_q)
print("🤖 Phi-4 (RAG):", rag_answer)

## 4. Wrap-Up & Best Practices
1. **Chain-of-Thought**: Great for debugging or certain QA tasks, but be mindful about revealing chain-of-thought to end users.
2. **RAG**: Use `azure-ai-inference` with retrieval results to ground your answers.
3. **OpenTelemetry**: Optionally integrate `opentelemetry-sdk` and `azure-core-tracing-opentelemetry` for full observability.
4. **Evaluate**: Use `azure-ai-evaluation` to measure your model’s performance.
5. **Cost & Performance**: Phi-4 aims to provide near GPT-4 performance at lower cost. Evaluate for your domain needs.

## 🎉 Congratulations!
You've seen how to:
- Use **Phi-4** with `AIProjectClient` and `azure-ai-inference`.
- Create a **chat** flow with chain-of-thought.
- Stub a **RAG** scenario.

> Happy hacking! 🏋️
